In [1]:
import numpy as np
import pandas as pd 
import scipy.stats
import gudhi as gd
import networkx as nx 
import random
from tqdm import tqdm

In [2]:
#extended version
def compute_persistent_homology(graph, filtration):
    """
    Compute the persistent homology of a graph with a given filtration.

    :param graph: A networkx graph.
    :param filtration: A list of node filtration values, corresponding to each node in the graph.
    :return: Persistent homology of dimension one.
    """
    # Create simplex tree
    st = gd.SimplexTree()

    # Add vertices with filtration values
    for i, node in enumerate(graph.nodes):
        st.insert([node], filtration=filtration[i])


    # Add edges
    for edge in graph.edges:
        st.insert(list(edge), filtration=max(filtration[list(graph.nodes()).index(edge[0])], filtration[list(graph.nodes()).index(edge[1])]))
    # Expand to clique complex
    st.extend_filtration()

    #romve null cycles
    persistence = st.extended_persistence(min_persistence=1e-5)
    tmp = []
    tmp.extend(persistence[0])
    tmp.extend(persistence[1])
    tmp.extend(persistence[2])
    tmp.extend(persistence[3])
    persistence = tmp

    #if birth > death change them
    tmp = []
    for gen in persistence:
        tmp.append((gen[0],(min(gen[1][0],gen[1][1]),max(gen[1][0],gen[1][1])) ))

    persistence = tmp

    #Filter for dimension one homology
    dim1_res = []
    for gen in persistence:
        if gen[0] == 1:
            if gen[1][0] != gen[1][1]:
                dim1_res.append(gen)
    return dim1_res


## limited nodes unweighted - given_th 

#### Complex

In [3]:
network_list = [
    'conf','email_eu_modified','hospital','school','work'
]
network_list = [
'email_eu_modified'
]

In [21]:
for network_name in network_list:
    for n_nodes_considered in tqdm([100,200,300,400,500,600,700,800,900]):
        #reading graph and simulations
        sims = pd.read_csv(f'../results/unweighted/given_th_n=300/unweighted_complex_{network_name}_EPH.csv')
        G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')

        #preprocessing
        # Create a mapping from current node names (str) to integers
        mapping = {node: int(float(node)) for node in G.nodes()}
        # Relabel the nodes in the graph using the mapping
        G = nx.relabel_nodes(G, mapping)


        #order of infection 
        order = sims.drop(['Unnamed: 0','theta','seed','q','EPH','corr'],axis=1)

        #limit n_nodes for orders
        order_considered = order[[str(i) for i in range(n_nodes_considered)]]
        order_prl_considered = order_considered.copy(deep=True)

        #fillna with max + 1
        order_considered = order_considered.apply(lambda row: row.fillna(row.max() + 1), axis=1)
        #for those with all nan 
        order_considered = order_considered.fillna(0)
        
        #limit n_nodes for graphs
        G_subgraph_considered = G.subgraph(np.array(G.nodes)[[i for i in range(n_nodes_considered)]])


        #EPH for each simulations
        EPH = []
        corr = [] 
        for row_index in range(len(order_considered)):
            #defining filtration
            filt = list(-order_considered.iloc[row_index].values)
            #compute EPH
            persistent_homology = compute_persistent_homology(graph=G_subgraph_considered,filtration=filt)
        
            #lifetime of genrators 
            life_set = []
            for gen in persistent_homology:
                life_set.append(gen[1][1] - gen[1][0])
            EPH.append(np.nanmean(life_set))
            
            #PRL
            deg_list = list(dict(G_subgraph_considered.degree).values())
            order_curr = list(order_prl_considered.iloc[row_index].values)

            try:
                rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
            except:
                # print('PRL method could\'nt find rho!!')
                rho = 0
                
            corr.append(rho)
            
        sims['EPH'+'node_lim'+str(n_nodes_considered)] = EPH
        sims['corr'+'node_lim'+str(n_nodes_considered)] = corr
        sims.to_csv(f'../results/unweighted/given_th_n=300/unweighted_complex_{network_name}_EPH_node_lim_{str(n_nodes_considered)}.csv')   

  0%|          | 0/9 [00:00<?, ?it/s]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_1253/763335258.py:43: RuntimeWarning: Mean of empty slice
  EPH.append(np.nanmean(life_set))
/opt/anaconda3/envs/tda/lib/python3.12/site-packages/scipy/stats/_mstats_basic.py:694: RuntimeWarning: invalid value encountered in divide
  t = rs * np.sqrt((dof / ((rs+1.0) * (1.0-rs))).clip(0))
 11%|█         | 1/9 [00:34<04:39, 34.91s/it]/opt/anaconda3/envs/tda/lib/python3.12/site-packages/scipy/stats/_mstats_basic.py:694: RuntimeWarning: invalid value encountered in divide
  t = rs * np.sqrt((dof / ((rs+1.0) * (1.0-rs))).clip(0))
/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_1253/763335258.py:43: RuntimeWarning: Mean of empty slice
  EPH.append(np.nanmean(life_set))
 22%|██▏       | 2/9 [03:57<15:33, 133.31s/it]/opt/anaconda3/envs/tda/lib/python3.12/site-packages/scipy/stats/_mstats_basic.py:694: RuntimeWarning: invalid value encountered in divide
  t = rs * np.sqrt((dof / ((rs+1.0)

KeyboardInterrupt: 

#### Simple

In [4]:
for network_name in network_list:
    for n_nodes_considered in tqdm([100,200,300,400,500,600,700,800,900]):

        #reading graph and simulations
        sims = pd.read_csv(f'../results/unweighted/given_th_n=300/unweighted_simple_{network_name}_EPH.csv')
        G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')

        #preprocessing
        # Create a mapping from current node names (str) to integers
        mapping = {node: int(float(node)) for node in G.nodes()}
        # Relabel the nodes in the graph using the mapping
        G = nx.relabel_nodes(G, mapping)


        #order of infection 
        order = sims.drop(['Unnamed: 0','betha','seed','EPH','corr'],axis=1)

        #limit n_nodes for orders
        order_considered = order[[str(i) for i in range(n_nodes_considered)]]
        order_prl_considered = order_considered.copy(deep=True)

        #fillna with max + 1
        order_considered = order_considered.apply(lambda row: row.fillna(row.max() + 1), axis=1)
        #for those with all nan 
        order_considered = order_considered.fillna(0)
        
        #limit n_nodes for graphs
        G_subgraph_considered = G.subgraph(np.array(G.nodes)[[i for i in range(n_nodes_considered)]])


        #EPH for each simulations
        EPH = []
        corr = [] 
        for row_index in range(len(order_considered)):
            #defining filtration
            filt = list(-order_considered.iloc[row_index].values)
            #compute EPH
            persistent_homology = compute_persistent_homology(graph=G_subgraph_considered,filtration=filt)
        
            #lifetime of genrators 
            life_set = []
            for gen in persistent_homology:
                life_set.append(gen[1][1] - gen[1][0])
            EPH.append(np.nanmean(life_set))
            
            #PRL
            deg_list = list(dict(G_subgraph_considered.degree).values())
            order_curr = list(order_prl_considered.iloc[row_index].values)

            try:
                rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
            except:
                # print('PRL method could\'nt find rho!!')
                rho = 0
                
            corr.append(rho)
            
        sims['EPH'+'node_lim'+str(n_nodes_considered)] = EPH
        sims['corr'+'node_lim'+str(n_nodes_considered)] = corr
        sims.to_csv(f'../results/unweighted/given_th_n=300/unweighted_simple_{network_name}_EPH_node_lim_{str(n_nodes_considered)}.csv')   

 89%|█████████████████████████████████████▎    | 8/9 [2:04:19<15:32, 932.44s/it]


KeyboardInterrupt: 

## limited steps unweighted - given_th 

#### Complex

In [4]:
for network_name in network_list:
    for threshold in tqdm(range(1,10)):
        #reading graph and simulations
        sims = pd.read_csv(f'../results/unweighted/given_th_n=300/unweighted_complex_{network_name}_EPH.csv')
        G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')
        
        #preprocessing
        mapping = {node: int(float(node)) for node in G.nodes()}
        G = nx.relabel_nodes(G, mapping)
        
        
        #order of infection 
        order = sims.drop(['Unnamed: 0','theta','seed','q','EPH','corr'],axis=1)
        #limit n_nodes for orders
        order = order[[str(i) for i in range(len(G.nodes()))]]
        order_prl_considered = order.copy(deep=True)
        order = order.apply(lambda row: row.fillna(row.max() + 1), axis=1)
        order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
        
        #EPH for each simulations
        EPH = []
        corr = [] 
        for row_index in range(len(order)):
            #defining filtration
            filt = list(-order.iloc[row_index].values)
            #compute EPH
            persistent_homology = compute_persistent_homology(graph=G,filtration=filt)
        
            #lifetime of genrators 
            life_set = []
            for gen in persistent_homology:
                life_set.append(gen[1][1] - gen[1][0])
            EPH.append(np.nanmean(life_set))
            
            #PRL
            deg_list = list(dict(G.degree).values())
            order_curr = list(order_prl_considered.iloc[row_index].values)
        
            try:
                rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
            except:
                # print('PRL method could\'nt find rho!!')
                rho = 0
                
            corr.append(rho)
            
        sims['EPH'+'step_lim'+str(threshold)] = EPH
        sims['corr'+'step_lim'+str(threshold)] = corr
        sims.to_csv(f'../results/unweighted/given_th_n=300/unweighted_complex_{network_name}_EPH_step_lim_{str(threshold)}.csv')
                

  0%|          | 0/9 [00:00<?, ?it/s]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/4270909276.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 11%|█         | 1/9 [01:15<10:06, 75.76s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/4270909276.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 22%|██▏       | 2/9 [02:35<09:07, 78.22s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/4270909276.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 33%|███▎      | 3/9 [03:57<08:00, 80.06s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/4270909276.py:18

#### Simple

In [6]:
for network_name in network_list:
    for threshold in tqdm(range(1,10)):
        #reading graph and simulations
        sims = pd.read_csv(f'../results/unweighted/given_th_n=300/unweighted_simple_{network_name}_EPH.csv')
        G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')
        
        #preprocessing
        mapping = {node: int(float(node)) for node in G.nodes()}
        G = nx.relabel_nodes(G, mapping)
        
        
        #order of infection 
        order = sims.drop(['Unnamed: 0','betha','seed','EPH','corr'],axis=1)
        #limit n_nodes for orders
        order = order[[str(i) for i in range(len(G.nodes()))]]
        order_prl_considered = order.copy(deep=True)
        order = order.apply(lambda row: row.fillna(row.max() + 1), axis=1)
        order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
        
        #EPH for each simulations
        EPH = []
        corr = [] 
        for row_index in range(len(order)):
            #defining filtration
            filt = list(-order.iloc[row_index].values)
            #compute EPH
            persistent_homology = compute_persistent_homology(graph=G,filtration=filt)
        
            #lifetime of genrators 
            life_set = []
            for gen in persistent_homology:
                life_set.append(gen[1][1] - gen[1][0])
            EPH.append(np.nanmean(life_set))
            
            #PRL
            deg_list = list(dict(G.degree).values())
            order_curr = list(order_prl_considered.iloc[row_index].values)
        
            try:
                rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
            except:
                # print('PRL method could\'nt find rho!!')
                rho = 0
                
            corr.append(rho)
            
        sims['EPH'+'step_lim'+str(threshold)] = EPH
        sims['corr'+'step_lim'+str(threshold)] = corr
        sims.to_csv(f'../results/unweighted/given_th_n=300/unweighted_simple_{network_name}_EPH_step_lim_{str(threshold)}.csv')
                

  0%|          | 0/9 [00:00<?, ?it/s]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/3955870615.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 11%|█         | 1/9 [00:14<01:59, 14.97s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/3955870615.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 22%|██▏       | 2/9 [00:31<01:51, 15.98s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/3955870615.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  order = order.applymap(lambda x: threshold if pd.isna(x) or x > threshold else x)
 33%|███▎      | 3/9 [00:48<01:37, 16.19s/it]/var/folders/r3/27v441rd1vb2qdthw2f3whr00000gn/T/ipykernel_52261/3955870615.py:18